# 07 Sequence Pipeline

Unsupervised sequence anomaly detection on Sysmon logs.

Each *process chain* (all logs for a given `process_guid`, sorted by `utc_time`) is
turned into sliding windows of consecutive log vectors.  Three autoencoder variants
are jointly searched over architecture and training hyperparameters with Optuna
(NAS + HPO).  Training is benign-only — the model learns a normal representation
of process behaviour.  Anomaly score = mean per-step reconstruction MSE of a window.

| Model | Encoder | Decoder | Layers |
|---|---|---|---|
| `lstm` | LSTM | LSTM | 1-6 (tuned by NAS+HPO) |
| `gru_lstm` | GRU | LSTM | 1-6 (tuned by NAS+HPO) |
| `transformer` | Transformer encoder | linear expand to [B, W, D] | 1-6 (tuned by NAS+HPO) |

Feature inputs: TF-IDF SVD (370 dims) and Word2Vec (352 dims), rule_name removed.


## 1. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import optuna
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
)
import joblib, json, gc, time, pickle
from pathlib import Path

optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED        = 42
CKPT_DIR    = Path('checkpoints')
SEQ_DIR     = CKPT_DIR / 'seq'
SEQ_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR  = CKPT_DIR / 'models' / 'seq'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device  : {DEVICE}')
print(f'CKPT_DIR: {CKPT_DIR}')
print(f'SEQ_DIR : {SEQ_DIR}')

Device  : cuda
CKPT_DIR: checkpoints
SEQ_DIR : checkpoints\seq


## 2. Feature Matrices

Load the norule variants (rule_name column removed) for both strategies.
Row order matches the original data loading order from notebook 05.


In [2]:
meta = json.loads((CKPT_DIR / 'meta.json').read_text())

strategies = {
    'tfidf_svd': {
        'X_b': np.load(CKPT_DIR / 'tfidf_svd' / 'X_b_norule.npy', mmap_mode='r'),
        'X_m': np.load(CKPT_DIR / 'tfidf_svd' / 'X_m_norule.npy', mmap_mode='r'),
        'input_dim': meta['feature_dim'] - 1,          # 370
    },
    'word2vec': {
        'X_b': np.load(CKPT_DIR / 'word2vec' / 'X_b_w2v_norule.npy', mmap_mode='r'),
        'X_m': np.load(CKPT_DIR / 'word2vec' / 'X_m_w2v_norule.npy', mmap_mode='r'),
        'input_dim': meta['strategy_b']['feature_dim']['w2v'] - 1,  # 352
    },
}

for name, s in strategies.items():
    print(f'{name:12s}  X_b={s["X_b"].shape}  X_m={s["X_m"].shape}  dim={s["input_dim"]}')

tfidf_svd     X_b=(1632903, 370)  X_m=(3447667, 370)  dim=370
word2vec      X_b=(1632903, 352)  X_m=(3447667, 352)  dim=352


## 3. Sequence Chain Index Extraction

Reload `process_guid` + `utc_time` from the raw data sources **in the same order** as
notebook 05 so that row _i_ in the feature matrix corresponds to row _i_ in the reloaded
DataFrame.

Result: `seq_chains_b.pkl` and `seq_chains_m.pkl` — lists of numpy integer arrays,
each array holding the row indices (into X_b or X_m) of one process chain, sorted
by `utc_time`.  Chains shorter than 2 events are discarded.

**Always run this cell** — it loads  /  into the kernel. If the pkl files already exist the raw data is not reloaded; only the pickle files are read.


In [3]:
_chains_b_path   = SEQ_DIR / 'seq_chains_b.pkl'
_chains_m_path   = SEQ_DIR / 'seq_chains_m.pkl'
_chains_m_lmd_path = SEQ_DIR / 'seq_chains_m_lmd.pkl'

if _chains_b_path.exists() and _chains_m_path.exists() and _chains_m_lmd_path.exists():
    print('Chain files already exist — loading.')
    with open(_chains_b_path, 'rb') as f:
        chains_b = pickle.load(f)
    with open(_chains_m_path, 'rb') as f:
        chains_m = pickle.load(f)
    with open(_chains_m_lmd_path, 'rb') as f:
        chains_m_lmd = pickle.load(f)
    print(f'chains_b:     {len(chains_b):,} chains')
    print(f'chains_m:     {len(chains_m):,} chains (all sources)')
    print(f'chains_m_lmd: {len(chains_m_lmd):,} chains (LMD-only)')
else:
    import sys, os
    sys.path.insert(0, os.path.abspath('..'))
    from data.ingest.lmd2023 import iter_chunks as lmd_iter
    from data.ingest.otrf import iter_compound, iter_atomic
    from data.ingest.splunk import iter_techniques as splunk_iter

    KEEP = ['process_guid', 'utc_time']

    # ── Benign: LMD benign-only ────────────────────────────────────────────
    print('Loading benign metadata...')
    _b_chunks = [c[KEEP] for c in lmd_iter(variant='2.3M', benign_only=True)]
    df_b_meta = pd.concat(_b_chunks, ignore_index=True)
    print(f'  {len(df_b_meta):,} benign rows')
    del _b_chunks

    # ── Malicious: LMD-mal + OTRF Atomic + OTRF Compound + Splunk ─────────
    print('Loading malicious metadata...')
    _mal_lmd_chunks = [c.loc[c['label'] > 0, KEEP]
                       for c in lmd_iter(variant='2.3M')
                       if (c['label'] > 0).any()]
    df_lmd_mal = pd.concat(_mal_lmd_chunks, ignore_index=True)

    _atomic_chunks  = [c[KEEP] for c in iter_atomic() if 'label' in c.columns and (c['label'] == 1).any()]
    df_atomic_meta  = pd.concat([c.loc[c['label'] == 1, KEEP] for c in iter_atomic()], ignore_index=True)                       if _atomic_chunks else pd.DataFrame(columns=KEEP)

    _compound_chunks = [c[KEEP] for c in iter_compound()]
    df_compound_meta = pd.concat(_compound_chunks, ignore_index=True) if _compound_chunks else pd.DataFrame(columns=KEEP)

    _splunk_chunks  = [c[KEEP] for c in splunk_iter()]
    df_splunk_meta  = pd.concat(_splunk_chunks, ignore_index=True) if _splunk_chunks else pd.DataFrame(columns=KEEP)

    df_m_meta = pd.concat([df_lmd_mal, df_atomic_meta, df_compound_meta, df_splunk_meta],
                           ignore_index=True)
    print(f'  {len(df_m_meta):,} malicious rows ({len(df_lmd_mal):,} LMD-only)')
    del _mal_lmd_chunks, _atomic_chunks, _compound_chunks, _splunk_chunks
    del df_atomic_meta, df_compound_meta, df_splunk_meta
    # Keep df_lmd_mal separate for unbiased chain extraction
    df_m_lmd_meta = df_lmd_mal.copy()
    del df_lmd_mal

    def _build_chains(df_meta):
        chains = []
        df_meta = df_meta.copy()
        df_meta['utc_time'] = pd.to_datetime(df_meta['utc_time'], errors='coerce', utc=True, format='mixed')
        for pid, grp in df_meta.groupby('process_guid', sort=False):
            grp_sorted = grp.sort_values('utc_time')
            idx = grp_sorted.index.values.astype(np.int32)
            if len(idx) >= 2:
                chains.append(idx)
        return chains

    print('Building benign chains...')
    chains_b = _build_chains(df_b_meta)
    print(f'  {len(chains_b):,} chains')

    print('Building malicious chains (all sources)...')
    chains_m = _build_chains(df_m_meta)
    print(f'  {len(chains_m):,} chains')

    print('Building malicious chains (LMD-only)...')
    chains_m_lmd = _build_chains(df_m_lmd_meta)
    print(f'  {len(chains_m_lmd):,} chains')

    del df_b_meta, df_m_meta, df_m_lmd_meta

    with open(_chains_b_path, 'wb') as f:
        pickle.dump(chains_b, f, protocol=4)
    with open(_chains_m_path, 'wb') as f:
        pickle.dump(chains_m, f, protocol=4)
    with open(_chains_m_lmd_path, 'wb') as f:
        pickle.dump(chains_m_lmd, f, protocol=4)
    print(f'Saved chains_b, chains_m, chains_m_lmd')

# Chain length statistics
_all_len_b   = np.array([len(c) for c in chains_b])
_all_len_m   = np.array([len(c) for c in chains_m])
_all_len_lmd = np.array([len(c) for c in chains_m_lmd])
print(f'\nBenign        chain length: median={np.median(_all_len_b):.0f}  p90={np.percentile(_all_len_b, 90):.0f}  max={_all_len_b.max()}')
print(f'Malicious(all) chain length: median={np.median(_all_len_m):.0f}  p90={np.percentile(_all_len_m, 90):.0f}  max={_all_len_m.max()}')
print(f'Malicious(LMD) chain length: median={np.median(_all_len_lmd):.0f}  p90={np.percentile(_all_len_lmd, 90):.0f}  max={_all_len_lmd.max()}')

Chain files already exist — loading.
chains_b:     4,065 chains
chains_m:     20,326 chains (all sources)
chains_m_lmd: 1,366 chains (LMD-only)

Benign        chain length: median=6  p90=336  max=1128737
Malicious(all) chain length: median=8  p90=90  max=1146251
Malicious(LMD) chain length: median=12  p90=602  max=244538


## 4. Model Definitions

### SequenceAE
- `lstm`: single-stack LSTM encoder → linear to latent → linear expand → LSTM decoder → output
- `gru_lstm`: GRU encoder → linear to latent → linear expand → LSTM decoder → output

### TransformerAE
- `input_proj` (linear) → `TransformerEncoder` → mean pool over W → `latent_fc` → `decode_fc`
  reshape to [B, W, D] → `output_proj`


In [4]:
class SequenceAE(nn.Module):
    """LSTM or GRU-LSTM sequence autoencoder."""
    def __init__(self, input_dim: int, hidden_dim: int, latent_dim: int,
                 n_layers: int, dropout: float, arch: str = 'lstm'):
        super().__init__()
        self.arch = arch
        rnn_cls_enc = nn.GRU  if arch == 'gru_lstm' else nn.LSTM
        rnn_cls_dec = nn.LSTM

        self.encoder_rnn = rnn_cls_enc(
            input_dim, hidden_dim, n_layers,
            batch_first=True, dropout=dropout if n_layers > 1 else 0.0,
        )
        self.enc_fc   = nn.Linear(hidden_dim, latent_dim)
        self.dec_fc   = nn.Linear(latent_dim, hidden_dim)
        self.decoder_rnn = rnn_cls_dec(
            hidden_dim, hidden_dim, n_layers,
            batch_first=True, dropout=dropout if n_layers > 1 else 0.0,
        )
        self.output_fc = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        # x: [B, W, D]
        _, enc_state = self.encoder_rnn(x)
        # Use the last layer's hidden state
        h = enc_state[0] if isinstance(enc_state, tuple) else enc_state
        # h: [n_layers, B, hidden_dim]
        h_last = h[-1]                          # [B, hidden_dim]
        z      = self.enc_fc(h_last)            # [B, latent_dim]
        h_dec  = self.dec_fc(z).unsqueeze(0)    # [1, B, hidden_dim]
        # Expand to n_layers
        h_dec  = h_dec.expand(self.decoder_rnn.num_layers, -1, -1).contiguous()
        c_dec  = torch.zeros_like(h_dec)
        dec_in = x.new_zeros(x.size(0), x.size(1), self.decoder_rnn.input_size)  # [B,W,hidden_dim]
        out, _ = self.decoder_rnn(dec_in, (h_dec, c_dec))
        return self.output_fc(out)              # [B, W, D]


class TransformerAE(nn.Module):
    """Transformer encoder → mean-pool → latent → linear decode."""
    def __init__(self, input_dim: int, hidden_dim: int, latent_dim: int,
                 n_layers: int, nhead: int, dropout: float):
        super().__init__()
        # Ensure hidden_dim is divisible by nhead
        if hidden_dim % nhead != 0:
            hidden_dim = max(nhead, (hidden_dim // nhead) * nhead)
        self.hidden_dim   = hidden_dim
        self.input_proj   = nn.Linear(input_dim, hidden_dim)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dim_feedforward=hidden_dim * 2,
            dropout=dropout, batch_first=True,
        )
        self.transformer  = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.latent_fc    = nn.Linear(hidden_dim, latent_dim)
        self.decode_fc    = nn.Linear(latent_dim, hidden_dim)
        self.output_proj  = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        # x: [B, W, D]
        h = self.input_proj(x)                  # [B, W, hidden_dim]
        h = self.transformer(h)                 # [B, W, hidden_dim]
        z = self.latent_fc(h.mean(dim=1))       # [B, latent_dim]
        d = self.decode_fc(z).unsqueeze(1)      # [B, 1, hidden_dim]
        d = d.expand(-1, x.size(1), -1)         # [B, W, hidden_dim]
        return self.output_proj(d)              # [B, W, D]


def build_model(params: dict, input_dim: int) -> nn.Module:
    arch = params['arch']
    if arch == 'transformer':
        nhead = params.get('nhead', 4)
        m = TransformerAE(
            input_dim   = input_dim,
            hidden_dim  = params['hidden_dim'],
            latent_dim  = params['latent_dim'],
            n_layers    = params['n_layers'],
            nhead       = nhead,
            dropout     = params['dropout'],
        )
    else:
        m = SequenceAE(
            input_dim  = input_dim,
            hidden_dim = params['hidden_dim'],
            latent_dim = params['latent_dim'],
            n_layers   = params['n_layers'],
            dropout    = params['dropout'],
            arch       = arch,
        )
    return m.to(DEVICE)

## 5. Training Utilities


In [5]:
import copy

VAL_BENIGN_RATIO = 0.95   # 95 / 5 production resampling


def load_X_b_ram(X_b, strategy_name: str) -> np.ndarray:
    """Load full benign feature matrix into RAM via one sequential read.
    At 2.4 GB on DDR5 this takes ~2 s and eliminates all mmap scatter-reads.
    """
    mb = X_b.shape[0] * X_b.shape[1] * 4 / 1e6
    print(f'  Loading {strategy_name} X_b into RAM ({mb:.0f} MB)...', flush=True)
    X_ram = np.array(X_b, dtype=np.float32)   # single sequential mmap read
    print(f'  Loaded  shape={X_ram.shape}', flush=True)
    return X_ram


class WindowDataset(torch.utils.data.Dataset):
    """Sliding-window dataset backed by a RAM-loaded feature matrix.

    X must be a contiguous float32 RAM array (from load_X_b_ram).
    __getitem__ is a pure RAM slice — no mmap, no disk, GPU-bound throughput.

    max_windows_per_chain: caps windows from any single chain to prevent
    one outlier chain (e.g. 1.1M events) from dominating training while
    all other chains remain fully represented.
    """
    def __init__(self, chains: list, X: np.ndarray, W: int, stride: int,
                 max_windows_per_chain: int = 2000, verbose: bool = False):
        self.X  = X
        self.W  = W
        rng     = np.random.default_rng(42)
        ci_list = []
        st_list = []
        capped  = 0
        for ci, chain in enumerate(chains):
            if len(chain) < W:
                continue
            starts = np.arange(0, len(chain) - W + 1, stride, dtype=np.int32)
            if len(starts) > max_windows_per_chain:
                idx    = rng.choice(len(starts), max_windows_per_chain, replace=False)
                starts = starts[idx]
                capped += 1
            ci_list.append(np.full(len(starts), ci, dtype=np.int32))
            st_list.append(starts)
        if not ci_list:
            raise ValueError(f'No windows built (W={W}).')
        self.ci_arr     = np.concatenate(ci_list)   # [N] chain index
        self.st_arr     = np.concatenate(st_list)   # [N] start position
        self.chain_list = chains
        if verbose:
            print(f'    WindowDataset: {len(self.ci_arr):,} windows '
                  f'({capped} chains capped at {max_windows_per_chain})', flush=True)

    def __len__(self) -> int:
        return len(self.ci_arr)

    def __getitem__(self, i: int) -> torch.Tensor:
        chain = self.chain_list[self.ci_arr[i]]
        start = self.st_arr[i]
        rows  = chain[start : start + self.W]   # [W] indices into X_ram
        win   = self.X[rows]                    # [W, D] — pure RAM lookup
        return torch.from_numpy(win)


def _build_val_windows(chains, X, W, stride, n_cap, rng):
    """Materialise a small capped window array for the validation set."""
    wins = []
    for ci in rng.permutation(len(chains)):
        chain = chains[ci]
        if len(chain) < W:
            continue
        for start in range(0, len(chain) - W + 1, stride):
            wins.append(X[chain[start : start + W]].astype(np.float32))
            if len(wins) >= n_cap:
                break
        if len(wins) >= n_cap:
            break
    if not wins:
        raise ValueError(f'No val windows built (W={W}).')
    return np.stack(wins, axis=0)


def build_val_set(chains_b, chains_m, X_b, X_m, W: int, stride: int,
                  val_size: int = 10_000, seed: int = SEED):
    """Build a materialised 95/5 benign/malicious validation window set."""
    rng   = np.random.default_rng(seed)
    n_ben = int(val_size * VAL_BENIGN_RATIO)
    n_mal = val_size - n_ben
    wins_b = _build_val_windows(chains_b, X_b, W, stride, n_ben, rng)
    wins_m = _build_val_windows(chains_m, X_m, W, stride, n_mal, rng)
    X_val  = np.concatenate([wins_b, wins_m], axis=0)
    y_val  = np.array([0] * len(wins_b) + [1] * len(wins_m), dtype=np.int8)
    return X_val, y_val


def seq_score(model: nn.Module, windows: np.ndarray,
              batch_size: int = 512) -> np.ndarray:
    """Mean per-step MSE reconstruction error for each window. Higher = more anomalous."""
    model.eval()
    scores = []
    t = torch.from_numpy(windows)
    with torch.no_grad():
        for i in range(0, len(t), batch_size):
            x   = t[i : i + batch_size].to(DEVICE)
            rec = model(x)
            mse = ((x - rec) ** 2).mean(dim=(1, 2)).cpu().numpy()
            scores.append(mse)
    return np.concatenate(scores)


def seq_evaluate(model: nn.Module, X_val: np.ndarray, y_val: np.ndarray,
                 model_name: str, strategy: str) -> dict:
    scores = seq_score(model, X_val)
    roc    = float(roc_auc_score(y_val, scores))
    ap     = float(average_precision_score(y_val, scores))
    prec, rec, thr = precision_recall_curve(y_val, scores)
    f1s    = 2 * prec * rec / (prec + rec + 1e-9)
    best   = int(np.argmax(f1s))
    f1     = float(f1s[best])
    thr_opt = float(thr[best]) if best < len(thr) else float(thr[-1])
    fpr_arr, tpr_arr, _ = roc_curve(y_val, scores)
    tpr_1fpr  = float(tpr_arr[np.searchsorted(fpr_arr, 0.01,  side='right') - 1])
    tpr_01fpr = float(tpr_arr[np.searchsorted(fpr_arr, 0.001, side='right') - 1])
    return {
        'model': model_name, 'strategy': strategy,
        'roc_auc': round(roc, 4), 'avg_precision': round(ap, 4),
        'f1': round(f1, 4),
        'tpr@1%fpr': round(tpr_1fpr, 4),
        'tpr@0.1%fpr': round(tpr_01fpr, 4),
        'threshold': round(thr_opt, 6),
    }


def train_sequence_ae(model: nn.Module,
                      train_ds: torch.utils.data.Dataset,
                      X_val:    np.ndarray,
                      y_val:    np.ndarray,
                      lr: float, weight_decay: float, batch_size: int,
                      n_epochs: int = 30, patience: int = 5) -> float:
    """Train on a WindowDataset (lazy mmap reads), validate on materialised val set."""
    optim  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=n_epochs)
    crit   = nn.MSELoss()
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                        num_workers=0,
                        pin_memory=(DEVICE.type == 'cuda'))

    best_ap    = 0.0
    best_state = None
    no_improv  = 0

    n_batches = len(loader)
    for epoch in range(1, n_epochs + 1):
        model.train()
        epoch_loss = 0.0
        for xb in loader:
            xb = xb.to(DEVICE)
            optim.zero_grad()
            rec  = model(xb)
            loss = crit(rec, xb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            epoch_loss += loss.item()
        sched.step()

        scores = seq_score(model, X_val)
        ap     = float(average_precision_score(y_val, scores))
        print(f'      epoch {epoch:3d}/{n_epochs}  loss={epoch_loss/n_batches:.4f}  AP={ap:.4f}', flush=True)
        if ap > best_ap:
            best_ap, best_state, no_improv = ap, copy.deepcopy(model.state_dict()), 0
        else:
            no_improv += 1
            if no_improv >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_ap


def _trial_callback(study, trial):
    roc  = trial.user_attrs.get('roc_auc',   'n/a')
    tpr1 = trial.user_attrs.get('tpr@1%fpr', 'n/a')
    arch = trial.params.get('arch', '?')
    strat= trial.user_attrs.get('strategy',  '?')
    W    = trial.params.get('window_size', '?')
    ap_s = f'{trial.value:.4f}' if trial.value is not None else 'FAILED'
    print(
        f'  trial {trial.number:3d} | {strat:10s} | arch={arch:10s} | W={W} '
        f'| AP={ap_s} | AUC={roc} | TPR@1%={tpr1}',
        flush=True,
    )

## 6. NAS + HPO

Optuna searches jointly over:
- `arch`: lstm / gru_lstm / transformer
- `window_size`: 5 - 50 (step 5)
- `stride`: 1, window_size//2, window_size (quarter, half, full step)
- `hidden_dim`, `n_layers`, `latent_dim`, `dropout`
- `lr`, `weight_decay`, `batch_size`

Objective: Average Precision on the 95/5 val set.


In [6]:
N_HPO_TRIALS = 60
N_HPO_EPOCHS = 10
HPO_PATIENCE = 2
HPO_DB       = SEQ_DIR / 'seq_hpo.db'

_ram_cache: dict = {}   # keyed by strategy_name -> X_ram (loaded once per kernel)

def _get_X_ram(strategy_name, X_b):
    if strategy_name not in _ram_cache:
        _ram_cache[strategy_name] = load_X_b_ram(X_b, strategy_name)
    return _ram_cache[strategy_name]


_best_hpo_ap: float = 0.0

def _seq_obj(trial: optuna.Trial) -> float:
    strategy_name = trial.suggest_categorical('strategy', ['tfidf_svd', 'word2vec'])
    s             = strategies[strategy_name]
    X_b, X_m      = s['X_b'], s['X_m']
    input_dim      = s['input_dim']

    arch       = trial.suggest_categorical('arch', ['lstm', 'gru_lstm', 'transformer'])
    W          = trial.suggest_int('window_size', 5, 50, step=5)
    stride_r   = trial.suggest_categorical('stride_ratio', [0.25, 0.5, 1.0])
    stride     = max(1, int(W * stride_r))
    hidden_dim = trial.suggest_categorical('hidden_dim', [64, 128, 256])
    n_layers   = trial.suggest_int('n_layers', 1, 6)
    latent_dim = trial.suggest_categorical('latent_dim', [32, 64, 128])
    dropout    = trial.suggest_float('dropout', 0.0, 0.3)
    lr         = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    wd         = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [256, 512, 1024])

    params = dict(arch=arch, hidden_dim=hidden_dim, n_layers=n_layers,
                  latent_dim=latent_dim, dropout=dropout)
    if arch == 'transformer':
        params['nhead'] = trial.suggest_categorical('nhead', [2, 4, 8])

    model = build_model(params, input_dim)

    print(f'  [trial {trial.number}] Building WindowDataset (W={W}, stride={stride})...', flush=True)
    train_ds     = WindowDataset(chains_b, X_b, W, stride, verbose=True)
    print(f'  [trial {trial.number}] Starting training ({len(train_ds):,} windows)...', flush=True)
    X_val, y_val = build_val_set(chains_b, chains_m_lmd, X_b, X_m, W, stride)

    ap = train_sequence_ae(model, train_ds, X_val, y_val,
                           lr=lr, weight_decay=wd, batch_size=batch_size,
                           n_epochs=N_HPO_EPOCHS, patience=HPO_PATIENCE)

    # Extra metrics for display
    scores = seq_score(model, X_val)
    roc    = float(roc_auc_score(y_val, scores))
    fpr_a, tpr_a, _ = roc_curve(y_val, scores)
    tpr1   = float(tpr_a[np.searchsorted(fpr_a, 0.01, side='right') - 1])
    trial.set_user_attr('roc_auc',   round(roc,  4))
    trial.set_user_attr('tpr@1%fpr', round(tpr1, 4))
    trial.set_user_attr('strategy',  strategy_name)

    global _best_hpo_ap
    if ap > _best_hpo_ap:
        _best_hpo_ap = ap
        _hpo_init_path = MODELS_DIR / 'seq_hpo_best_init.pt'
        torch.save({k: v.cpu() for k, v in model.state_dict().items()}, _hpo_init_path)
        print(f'  → New best AP={ap:.4f}, init weights saved', flush=True)

    del model, train_ds, X_val
    gc.collect()
    torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
    return ap


storage  = f'sqlite:///{HPO_DB.as_posix()}'
study_seq = optuna.create_study(
    study_name    = 'seq_nas_hpo',
    storage       = storage,
    direction     = 'maximize',
    load_if_exists= True,
    sampler       = optuna.samplers.TPESampler(seed=SEED),
    pruner        = optuna.pruners.MedianPruner(n_startup_trials=5),
)

_best_hpo_ap = 0.0
print(f'Starting NAS+HPO — {N_HPO_TRIALS} trials ({len(study_seq.trials)} already done)')
study_seq.optimize(
    _seq_obj,
    n_trials   = N_HPO_TRIALS,
    callbacks  = [_trial_callback],
    catch       = (Exception,),
)

best_trial = study_seq.best_trial
print(f'\nBest trial: {best_trial.number}  AP={best_trial.value:.4f}')
print(json.dumps(best_trial.params, indent=2))

seq_best_path = MODELS_DIR / 'best_params.json'
seq_best_path.write_text(json.dumps({
    'trial'    : best_trial.number,
    'ap'       : best_trial.value,
    'roc_auc'  : best_trial.user_attrs.get('roc_auc'),
    'tpr@1%fpr': best_trial.user_attrs.get('tpr@1%fpr'),
    'strategy' : best_trial.user_attrs.get('strategy'),
    'params'   : best_trial.params,
}, indent=2))
print(f'Saved best params → {seq_best_path}')

Starting NAS+HPO — 60 trials (0 already done)
  [trial 0] Building WindowDataset (W=10, stride=5)...
    WindowDataset: 51,731 windows (2 chains capped at 2000)
  [trial 0] Starting training (51,731 windows)...
      epoch   1/10  loss=1.3917  AP=0.2121
      epoch   2/10  loss=1.2116  AP=0.2086
      epoch   3/10  loss=1.2134  AP=0.2060
  → New best AP=0.2121, init weights saved
  trial   0 | word2vec   | arch=lstm       | W=10 | AP=0.2121 | AUC=0.8275 | TPR@1%=0.034
  [trial 1] Building WindowDataset (W=30, stride=30)...
    WindowDataset: 11,561 windows (2 chains capped at 2000)
  [trial 1] Starting training (11,561 windows)...
      epoch   1/10  loss=2.4005  AP=0.6635
      epoch   2/10  loss=1.1682  AP=0.6712
      epoch   3/10  loss=0.9441  AP=0.6726
      epoch   4/10  loss=0.9339  AP=0.6737
      epoch   5/10  loss=0.9270  AP=0.6735
      epoch   6/10  loss=0.9185  AP=0.6772
      epoch   7/10  loss=0.9202  AP=0.6758
      epoch   8/10  loss=0.9254  AP=0.6733
  → New best AP=0

## 7. Retrain Best + Evaluate

Load best params from disk (safe after kernel restart), train for more epochs on
the full benign training set, and evaluate on a fresh held-out test set.


In [7]:
_seq_best_path = MODELS_DIR / 'best_params.json'
if not _seq_best_path.exists():
    raise FileNotFoundError(
        f'{_seq_best_path} not found — run the NAS+HPO cell first.'
    )

_best = json.loads(_seq_best_path.read_text())
seq_best_params = _best['params']
seq_best_strategy = _best.get('strategy', seq_best_params.get('strategy'))

print(f'Best strategy : {seq_best_strategy}')
print(f'Best params   : {json.dumps(seq_best_params, indent=2)}')
print(f'HPO AP        : {_best["ap"]:.4f}')

s_best     = strategies[seq_best_strategy]
X_b_best   = s_best['X_b']
X_m_best   = s_best['X_m']
input_dim  = s_best['input_dim']

W      = seq_best_params['window_size']
sr     = seq_best_params['stride_ratio']
stride = max(1, int(W * sr))

print(f'\nLoading X_b_best into RAM for final retrain...')
X_ram = load_X_b_ram(X_b_best, seq_best_strategy)
train_ds = WindowDataset(chains_b, X_ram, W, stride, verbose=True)
print(f'Training windows: {len(train_ds):,}')

print('Building test val set...')
X_val, y_val = build_val_set(chains_b, chains_m_lmd, X_ram, X_m_best,
                              W, stride, val_size=20_000, seed=SEED)
print(f'Val/test windows : {X_val.shape}  (pos={y_val.sum():,})')

model_best = build_model(seq_best_params, input_dim)
_hpo_init_path = MODELS_DIR / 'seq_hpo_best_init.pt'
if _hpo_init_path.exists():
    model_best.load_state_dict(torch.load(_hpo_init_path, map_location=DEVICE))
    print('Warm-started from best HPO trial weights')

print('\nRetraining with full epochs...')
_t0 = time.time()
best_ap = train_sequence_ae(
    model_best, train_ds, X_val, y_val,
    lr           = seq_best_params['lr'],
    weight_decay = seq_best_params['weight_decay'],
    batch_size   = seq_best_params['batch_size'],
    n_epochs     = 60,
    patience     = 15,
)
print(f'Done in {time.time() - _t0:.0f}s  best_ap={best_ap:.4f}')

result_seq = seq_evaluate(model_best, X_val, y_val,
                          model_name=f'SeqAE({seq_best_params["arch"]})',
                          strategy=seq_best_strategy)
print('\n=== Sequence Model Results ===')
for k, v in result_seq.items():
    print(f'  {k:20s}: {v}')

# Save model weights
_wt_path = MODELS_DIR / f'seq_ae_{seq_best_strategy}.pt'
torch.save(model_best.state_dict(), _wt_path)
print(f'Weights saved → {_wt_path}')

# Append to results JSON
_prev_path = CKPT_DIR / 'models' / 'unsupervised_results.json'
seq_results = []
if _prev_path.exists():
    seq_results = json.loads(_prev_path.read_text())

# Remove old seq results for same strategy before appending
seq_results = [r for r in seq_results if not (
    'SeqAE' in r.get('model', '') and r.get('strategy') == seq_best_strategy
)]
seq_results.append(result_seq)
_seq_res_path = MODELS_DIR / 'seq_results.json'
_seq_res_path.write_text(json.dumps(seq_results, indent=2))
print(f'Results saved → {_seq_res_path}')

Best strategy : word2vec
Best params   : {
  "strategy": "word2vec",
  "arch": "transformer",
  "window_size": 45,
  "stride_ratio": 1.0,
  "hidden_dim": 256,
  "n_layers": 2,
  "latent_dim": 128,
  "dropout": 0.018647794316122058,
  "lr": 0.0019603157690246086,
  "weight_decay": 2.2066719973633085e-06,
  "batch_size": 256,
  "nhead": 8
}
HPO AP        : 0.9110

Loading X_b_best into RAM for final retrain...
  Loading word2vec X_b into RAM (2299 MB)...
  Loaded  shape=(1632903, 352)
    WindowDataset: 8,816 windows (2 chains capped at 2000)
Training windows: 8,816
Building test val set...
Val/test windows : (20000, 45, 352)  (pos=1,000)
Warm-started from best HPO trial weights

Retraining with full epochs...
      epoch   1/60  loss=0.2022  AP=0.9564
      epoch   2/60  loss=0.0979  AP=0.9532
      epoch   3/60  loss=0.0910  AP=0.9537
      epoch   4/60  loss=0.0870  AP=0.9456
      epoch   5/60  loss=0.0835  AP=0.9415
      epoch   6/60  loss=0.0816  AP=0.9486
      epoch   7/60  loss

## 8. Results Summary


In [8]:
import json
from pathlib import Path

_prev_path = CKPT_DIR / 'models' / 'unsupervised_results.json'
_seq_path  = CKPT_DIR / 'models' / 'seq' / 'seq_results.json'

all_results = []
if _prev_path.exists():
    all_results += json.loads(_prev_path.read_text())
if _seq_path.exists():
    all_results += [r for r in json.loads(_seq_path.read_text())
                    if 'SeqAE' in r.get('model', '')]

if not all_results:
    print('No results found — run retrain cells first.')
else:
    df_res = pd.DataFrame(all_results)
    cols   = ['model', 'strategy', 'roc_auc', 'avg_precision', 'f1',
               'tpr@1%fpr', 'tpr@0.1%fpr']
    cols   = [c for c in cols if c in df_res.columns]
    df_res = df_res[cols].sort_values('avg_precision', ascending=False).reset_index(drop=True)
    print(df_res.to_string(index=False))

             model  strategy  roc_auc  avg_precision     f1  tpr@1%fpr  tpr@0.1%fpr
SeqAE(transformer)  word2vec   0.9869         0.9564 0.9443     0.9720       0.9090
       Autoencoder  word2vec   0.9895         0.7722 0.8213     0.4283       0.0771
       Autoencoder tfidf_svd   0.9838         0.6684 0.8004     0.1315       0.1205
   MiniBatchKMeans  word2vec   0.9872         0.6667 0.7808     0.3542       0.0183
   IsolationForest  word2vec   0.9629         0.5352 0.7120     0.1317       0.0372
   MiniBatchKMeans tfidf_svd   0.9772         0.5279 0.7536     0.1101       0.0162
   IsolationForest tfidf_svd   0.9471         0.4647 0.5617     0.1314       0.0241
    SGDOneClassSVM tfidf_svd   0.8902         0.2035 0.4020     0.0247       0.0016
    SGDOneClassSVM  word2vec   0.8589         0.1680 0.3938     0.0304       0.0022
